# Validacion + evaluacion RAGAS de las respuestas RAFT (modelo local)

Evalua **solo las 3 combinaciones nuevas** de `resultados_generacion_llm_raft.json`: `openai_cs800__raft-llama3.1-8b__{raft, estructurado, directo}`. Las 8 combinaciones de Claude ya estan evaluadas y no se tocan. Al final solo se **leen** sus resultados existentes para la tabla comparativa.

**Parte A – Validacion (sin API, gratis).** Clasifica cada respuesta RAFT:

| Tipo | Que detecta | Que se hace |
|---|---|---|
| **Falla tecnica** | respuesta vacia o `None`, contexto truncado por Ollama, ids duplicados, preguntas distintas a las 196 de la evaluacion original, contextos o referencia vacios | **Bloquea** la evaluacion: hay que regenerar esa respuesta. No es culpa del modelo. |
| **Falla del modelo** | texto degenerado (bucles), respuesta cortada por longitud, formato pedido no respetado (`Respuesta final:` / `##Answer:`) | **Se marca y se evalua igual.** Es un error real del modelo y cuenta en su puntaje. |
| **Informativo** | abstencion ("no se encuentra en el contexto"), citas RAFT no literales | Solo se reporta. |

**Por que no se descartan las fallas del modelo.** Quitar las respuestas malas inflaria el puntaje de RAFT frente a Claude. El resultado principal incluye todo. Como **analisis de sensibilidad**, la tabla comparativa final tambien se calcula excluyendo esas preguntas **en todas las combinaciones**, incluidas las de Claude. Asi la comparacion sigue siendo pareada.

**Parte B – Evaluacion RAGAS.** Mismas 4 metricas, mismo juez (`openai/gpt-4.1-mini` con los mismos respaldos) y mismos embeddings que la evaluacion original, para que los numeros sean comparables.

**Salidas nuevas** en `./validacion/` (no se sobrescribe nada existente):
- `validacion_respuestas_raft.csv`: una fila por pregunta × combinacion RAFT, con las marcas.
- `checkpoints_evaluacion_generacion/openai_cs800__raft-llama3.1-8b__*.csv`: checkpoints de RAGAS.
- `evaluacion_generacion_detalle_raft.csv`: metricas por pregunta, con el mismo formato del detalle original mas las marcas al final.
- `evaluacion_generacion_resumen_raft.csv`: promedios por combinacion RAFT.
- `comparacion_openai_cs800_con_raft.csv`: Claude (resultados existentes) frente a RAFT, con todas las respuestas y con la exclusion pareada.

**Variables de entorno:** `OPENROUTER_API_KEY` (solo para la Parte B).

In [ ]:
%pip install -q ragas langchain-openai pandas

In [1]:
import sys
import types

# Mismo parche del notebook original (import roto de langchain_community en ragas)
if "langchain_community.chat_models.vertexai" not in sys.modules:
    try:
        import langchain_community.chat_models.vertexai  # noqa: F401
    except ModuleNotFoundError:
        modulo_falso = types.ModuleType("langchain_community.chat_models.vertexai")

        class ChatVertexAI:  # stub, nunca se instancia de verdad
            pass

        modulo_falso.ChatVertexAI = ChatVertexAI
        sys.modules["langchain_community.chat_models.vertexai"] = modulo_falso
        print("Aviso: se aplico un parche al import roto de langchain_community.chat_models.vertexai.")

Aviso: se aplico un parche al import roto de langchain_community.chat_models.vertexai.


C:\Users\gomex\AppData\Local\Temp\ipykernel_32592\3660541490.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community.chat_models.vertexai  # noqa: F401


# Parte A – Validacion

## A1. Cargar las 3 combinaciones RAFT

In [2]:
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd

RUTA_GENERACION = "./validacion/resultados_generacion_llm_raft.json"        # solo RAFT
RUTA_GENERACION_ORIGINAL = "./validacion/resultados_generacion_llm.json"   # solo para verificar las preguntas
RUTA_VALIDACION_CSV = "./validacion/validacion_respuestas_raft.csv"

with open(RUTA_GENERACION, "r", encoding="utf-8") as f:
    generacion = json.load(f)
combinaciones_generadas = [c for c in generacion["combinaciones"] if c["modelo_llm"].startswith("raft")]


def etiqueta_de(c):
    return f"{c['pipeline_retrieve']}__{c['modelo_llm']}__{c['estilo_prompt']}"


print(f"Combinaciones RAFT cargadas: {len(combinaciones_generadas)}")
for c in combinaciones_generadas:
    print(f"  {etiqueta_de(c):55s} -> {c['total_preguntas']} respuestas")

Combinaciones RAFT cargadas: 3
  openai_cs800__raft-llama3.1-8b__raft                    -> 196 respuestas
  openai_cs800__raft-llama3.1-8b__estructurado            -> 196 respuestas
  openai_cs800__raft-llama3.1-8b__directo                 -> 196 respuestas


## A2. Reglas de validacion

Los umbrales de "texto degenerado" son conservadores. Por ejemplo, una tabla larga que repite "Piso 1 … Piso 2 …" **no** se marca como bucle; solo los bucles reales.

In [3]:
# Umbrales
MIN_PALABRAS_DEGENERADO = 40
REPETICION_NGRAMA = 8          # tamano del n-grama de palabras
REPETICION_MAX_VECES = 8       # si un mismo 8-grama aparece >= 8 veces -> bucle
DIVERSIDAD_MINIMA = 0.20       # palabras unicas / palabras totales (con >= 150 palabras)

PATRON_ABSTENCION = re.compile(
    r"no se encuentra|no (contiene|incluye|proporciona|menciona|especifica)|no hay informaci|no es posible (determinar|responder)",
    re.IGNORECASE,
)


def norm_literal(s):
    s = unicodedata.normalize("NFKC", s or "").lower()
    return re.sub(r"[\W_]+", "", re.sub(r"<br\s*/?>", " ", s))


def es_degenerado(texto):
    palabras = (texto or "").split()
    if len(palabras) < MIN_PALABRAS_DEGENERADO:
        return False
    ngramas = Counter(tuple(palabras[i:i + REPETICION_NGRAMA]) for i in range(len(palabras) - REPETICION_NGRAMA))
    if ngramas and ngramas.most_common(1)[0][1] >= REPETICION_MAX_VECES:
        return True
    return len(palabras) >= 150 and len(set(palabras)) / len(palabras) < DIVERSIDAD_MINIMA


def formato_respetado(cruda, estilo):
    if estilo == "estructurado":
        return "Respuesta final:" in (cruda or "")
    if estilo == "raft":
        return "##Answer:" in (cruda or "")
    return None   # 'directo' no pide formato


def citas_raft(cruda, contextos):
    citas = re.findall(r"##begin_quote##(.*?)##end_quote##", cruda or "", flags=re.S)
    ctx = norm_literal(" ".join(contextos or []))
    literales = sum(1 for q in citas if norm_literal(q) and norm_literal(q) in ctx)
    return len(citas), literales

## A3. Aplicar las reglas

In [8]:
# Las 196 preguntas con las que se evaluaron las combinaciones openai_cs800 de Claude
with open(RUTA_GENERACION_ORIGINAL, "r", encoding="utf-8") as f:
    original = json.load(f)
preguntas_originales = {
    r["pregunta"] for c in original["combinaciones"] if c["pipeline_retrieve"] == "openai_cs800" for r in c["resultados"]
}
del original

filas = []
for c in combinaciones_generadas:
    etiqueta = etiqueta_de(c)
    vistos = Counter(r["id"] for r in c["resultados"])
    for r in c["resultados"]:
        cruda = r.get("respuesta_generada_cruda")
        resp = r.get("respuesta_generada")
        n_citas, n_literales = citas_raft(cruda, r.get("contextos_usados")) if c["estilo_prompt"] == "raft" else (None, None)
        filas.append({
            "combinacion": etiqueta,
            "pipeline_retrieve": c["pipeline_retrieve"],
            "modelo_llm": c["modelo_llm"],
            "estilo_prompt": c["estilo_prompt"],
            "id": r["id"],
            "pregunta": r["pregunta"],
            # --- fallas tecnicas ---
            "tec_respuesta_vacia": not (resp or "").strip(),
            "tec_contexto_truncado": bool(r.get("contexto_posiblemente_truncado")),
            "tec_sin_contextos": not r.get("contextos_usados"),
            "tec_sin_referencia": not (r.get("respuesta_esperada") or "").strip(),
            "tec_id_duplicado": vistos[r["id"]] > 1,
            "tec_pregunta_no_evaluada_en_claude": r["pregunta"] not in preguntas_originales,
            # --- fallas del modelo ---
            "mod_degenerado": es_degenerado(cruda),
            "mod_cortada_por_longitud": bool(r.get("respuesta_cortada_por_longitud")),
            "mod_formato_no_respetado": formato_respetado(cruda, c["estilo_prompt"]) is False,
            # --- informativo ---
            "info_abstencion": bool(PATRON_ABSTENCION.search(resp or "")),
            "info_citas": n_citas,
            "info_citas_literales": n_literales,
            "info_palabras_respuesta": len((resp or "").split()),
        })

val = pd.DataFrame(filas)
cols_tec = [c for c in val.columns if c.startswith("tec_")]
cols_mod = [c for c in val.columns if c.startswith("mod_")]
val["falla_tecnica"] = val[cols_tec].any(axis=1)
val["falla_modelo"] = val[cols_mod].any(axis=1)

# Cada combinacion RAFT debe tener exactamente las mismas 196 preguntas que Claude
faltantes = {e: len(preguntas_originales - set(g)) for e, g in val.groupby("combinacion")["pregunta"]}
print("Preguntas que faltan en cada combinacion RAFT:", faltantes)

val.to_csv(RUTA_VALIDACION_CSV, index=False, encoding="utf-8")
print(f"Detalle de validacion guardado en {RUTA_VALIDACION_CSV}")

Preguntas que faltan en cada combinacion RAFT: {'openai_cs800__raft-llama3.1-8b__directo': 0, 'openai_cs800__raft-llama3.1-8b__estructurado': 0, 'openai_cs800__raft-llama3.1-8b__raft': 0}
Detalle de validacion guardado en ./validacion/validacion_respuestas_raft.csv


## A4. Resumen de la validacion

In [5]:
resumen_val = val.groupby("combinacion").agg(
    n=("id", "size"),
    fallas_tecnicas=("falla_tecnica", "sum"),
    degeneradas=("mod_degenerado", "sum"),
    cortadas=("mod_cortada_por_longitud", "sum"),
    sin_formato=("mod_formato_no_respetado", "sum"),
    fallas_modelo=("falla_modelo", "sum"),
    abstenciones=("info_abstencion", "sum"),
    palabras_mediana=("info_palabras_respuesta", "median"),
)
citas = val[val["estilo_prompt"] == "raft"].groupby("combinacion")[["info_citas", "info_citas_literales"]].sum()
resumen_val = resumen_val.join(citas)
resumen_val["pct_citas_literales"] = (100 * resumen_val["info_citas_literales"] / resumen_val["info_citas"]).round(1)
resumen_val

,n,fallas_tecnicas,degeneradas,cortadas,sin_formato,fallas_modelo,abstenciones,palabras_mediana,info_citas,info_citas_literales,pct_citas_literales
combinacion,,,,,,,,,,,
openai_cs800__raft-llama3.1-8b__directo,196,0,0,1,0,1,3,36.0,NaN,NaN,NaN
openai_cs800__raft-llama3.1-8b__estructurado,196,0,0,0,0,0,3,27.0,NaN,NaN,NaN
openai_cs800__raft-llama3.1-8b__raft,196,0,1,2,2,2,0,25.0,371.0,271.0,73.0


**Como leer la tabla:**
- `fallas_tecnicas` debe ser 0 en todas las filas. Si no lo es, la siguiente celda se detiene y lista que regenerar.
- `fallas_modelo` cuenta los errores reales de cada modelo. Reportalos en la tesis junto a las metricas.
- `abstenciones`: el paper RAFT entrena para responder incluso cuando el documento correcto no esta en el contexto (fraccion 1−P). Por eso se espera que el modelo RAFT se abstenga menos que Claude (Claude: 11–20 de 196), y esto puede afectar su `faithfulness`.
- `pct_citas_literales` (solo estilo `raft`): porcentaje de citas `##begin_quote##` que existen textualmente en los 5 contextos. Mide que tanto se apoya el razonamiento en el documento.

In [9]:
tecnicas = val[val["falla_tecnica"]]
if len(tecnicas) or any(faltantes.values()):
    print("Hay fallas TECNICAS. Regenera estas respuestas antes de evaluar:")
    display(tecnicas[["combinacion", "id", "pregunta"] + [c for c in cols_tec if tecnicas[c].any()]])
    raise RuntimeError(f"{len(tecnicas)} respuestas con fallas tecnicas (ver tabla).")
print("Sin fallas tecnicas: las respuestas RAFT son evaluables y comparables 1 a 1 con Claude.")

# Ejemplos de fallas del modelo, para revisar a mano
pd.set_option("display.max_colwidth", 200)
fallas = val[val["falla_modelo"]][["combinacion", "id", "pregunta"] + cols_mod]
print(f"\n{len(fallas)} respuestas con fallas del modelo (se evaluan igual):")
fallas

Sin fallas tecnicas: las respuestas RAFT son evaluables y comparables 1 a 1 con Claude.

3 respuestas con fallas del modelo (se evaluan igual):


,combinacion,id,pregunta,mod_degenerado,mod_cortada_por_longitud,mod_formato_no_respetado
10,openai_cs800__raft-llama3.1-8b__raft,39,"Según el documento, ¿qué factores necesitan mayor fortalecimiento según la Figura 5?",False,True,True
82,openai_cs800__raft-llama3.1-8b__raft,293,¿Cuáles son los campos clave que se mencionan como fundamentales y se integran en el programa de Ingeniería en Sistemas?,True,True,True
402,openai_cs800__raft-llama3.1-8b__directo,39,"Según el documento, ¿qué factores necesitan mayor fortalecimiento según la Figura 5?",False,True,False


In [10]:
# Ver la salida completa de una respuesta marcada (cambia el indice para recorrerlas)
INDICE = 0
if len(fallas):
    f = fallas.iloc[INDICE]
    c = next(c for c in combinaciones_generadas if etiqueta_de(c) == f["combinacion"])
    r = next(r for r in c["resultados"] if r["id"] == f["id"])
    print(f["combinacion"], "| id", f["id"])
    print("PREGUNTA:", r["pregunta"])
    print("ESPERADA:", r["respuesta_esperada"])
    print("CRUDA (ultimos 1500 caracteres):\n", (r["respuesta_generada_cruda"] or "")[-1500:])

openai_cs800__raft-llama3.1-8b__raft | id 39
PREGUNTA: Según el documento, ¿qué factores necesitan mayor fortalecimiento según la Figura 5?
ESPERADA: Según la Figura 5, los factores relacionados con el bienestar de la comunidad académica (Factor 9) y los recursos físicos y tecnológicos (Factor 12), aunque clasificados en un Alto Grado, necesitan un mayor fortalecimiento para alcanzar el óptimo ideal.
CRUDA (ultimos 1500 caracteres):
 quote##FACTOR 37##end_quote##, ##begin_quote##FACTOR 38##end_quote##, ##begin_quote##FACTOR 39##end_quote##, ##begin_quote##FACTOR 40##end_quote##, ##begin_quote##FACTOR 41##end_quote##, ##begin_quote##FACTOR 42##end_quote##, ##begin_quote##FACTOR 43##end_quote##, ##begin_quote##FACTOR 44##end_quote##, ##begin_quote##FACTOR 45##end_quote##, ##begin_quote##FACTOR 46##end_quote##, ##begin_quote##FACTOR 47##end_quote##, ##begin_quote##FACTOR 48##end_quote##, ##begin_quote##FACTOR 49##end_quote##, ##begin_quote##FACTOR 50##end_quote##, ##begin_quote##FACTOR 51

## A5. Preguntas para el analisis de sensibilidad (exclusion pareada)

Si una pregunta tiene una falla del modelo en cualquier combinacion RAFT, en la tabla de sensibilidad se excluye de **todas** las combinaciones, tambien de las de Claude. El resultado principal no excluye nada.

In [11]:
preguntas_excluir_sensibilidad = set(val.loc[val["falla_modelo"], "pregunta"])
print(f"Preguntas excluidas en el analisis de sensibilidad: {len(preguntas_excluir_sensibilidad)} "
      f"de {len(preguntas_originales)} -> quedan {len(preguntas_originales) - len(preguntas_excluir_sensibilidad)}")

Preguntas excluidas en el analisis de sensibilidad: 2 de 196 -> quedan 194


# Parte B – Evaluacion RAGAS

## B1. Juez, embeddings y metricas

In [12]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from ragas.metrics import Faithfulness, ResponseRelevancy, AnswerCorrectness, SemanticSimilarity

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    try:
        from dotenv import load_dotenv
        load_dotenv("./.env")
        OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
if not OPENROUTER_API_KEY:
    raise RuntimeError("Define la variable de entorno OPENROUTER_API_KEY")

MODELO_JUEZ = "openai/gpt-4.1-mini"
MODELOS_JUEZ_RESPALDO = ["google/gemini-2.5-flash", "deepseek/deepseek-chat"]

llm_juez_base = ChatOpenAI(
    model=MODELO_JUEZ,
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    temperature=0,
    extra_body={"models": MODELOS_JUEZ_RESPALDO},
)
evaluator_llm = LangchainLLMWrapper(llm_juez_base)

embeddings_base = OpenAIEmbeddings(
    model="openai/text-embedding-3-large",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings_base)

run_config = RunConfig(timeout=120, max_retries=8, max_wait=90, max_workers=4)

METRICAS = [
    Faithfulness(llm=evaluator_llm),
    ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    AnswerCorrectness(llm=evaluator_llm, embeddings=evaluator_embeddings),
    SemanticSimilarity(embeddings=evaluator_embeddings),
]
print("Metricas a calcular:", [m.name for m in METRICAS])

c:\Users\gomex\OneDrive\Escritorio\TG - Pruebas 2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\gomex\AppData\Local\Temp\ipykernel_32592\216063593.py:6: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy, AnswerCorrectness, SemanticSimilarity
C:\Users\gomex\AppData\Local\Temp\ipykernel_32592\216063593.py:6: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import Faithfulness, ResponseR

Metricas a calcular: ['faithfulness', 'answer_relevancy', 'answer_correctness', 'semantic_similarity']


C:\Users\gomex\AppData\Local\Temp\ipykernel_32592\216063593.py:29: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(llm_juez_base)
C:\Users\gomex\AppData\Local\Temp\ipykernel_32592\216063593.py:36: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings_base)


## B2. Evaluar las 3 combinaciones RAFT (con checkpoint)

Con `TAMANO_MUESTRA = 15` haces una prueba barata. Esos checkpoints se guardan con el sufijo `__muestra15`, asi que **no** contaminan la corrida completa (en el notebook original si lo hacian). Para la corrida completa, vuelve a `None`.

In [13]:
import time
from ragas import evaluate, EvaluationDataset

TAMANO_MUESTRA = None

RUTA_CHECKPOINTS = Path("./validacion/checkpoints_evaluacion_generacion")
RUTA_CHECKPOINTS.mkdir(parents=True, exist_ok=True)

RUTA_SALIDA_DETALLE = "./validacion/evaluacion_generacion_detalle_raft.csv"
RUTA_SALIDA_RESUMEN = "./validacion/evaluacion_generacion_resumen_raft.csv"
RUTA_SALIDA_COMPARACION = "./validacion/comparacion_openai_cs800_con_raft.csv"
RUTA_DETALLE_CLAUDE = "./validacion/evaluacion_generacion_detalle.csv"   # existente, solo lectura


def construir_dataset(resultados, tamano_muestra=None):
    filas = resultados[:tamano_muestra] if tamano_muestra else resultados
    filas = [
        {
            "user_input": r["pregunta"],
            "response": r.get("respuesta_generada") or "",
            "retrieved_contexts": r.get("contextos_usados") or [],
            "reference": r.get("respuesta_esperada") or "",
        }
        for r in filas
    ]
    return EvaluationDataset.from_list(filas)


def evaluar_combinacion(etiqueta, resultados, intentos_externos=3, espera_entre_intentos=60):
    sufijo = f"__muestra{TAMANO_MUESTRA}" if TAMANO_MUESTRA else ""
    ruta_checkpoint = RUTA_CHECKPOINTS / f"{etiqueta}{sufijo}.csv"

    if ruta_checkpoint.exists():
        print(f"  [{etiqueta}] ya evaluado antes, se reutiliza el checkpoint.")
        return pd.read_csv(ruta_checkpoint)

    dataset = construir_dataset(resultados, TAMANO_MUESTRA)
    for intento in range(1, intentos_externos + 1):
        try:
            resultado = evaluate(
                dataset=dataset,
                metrics=METRICAS,
                llm=evaluator_llm,
                embeddings=evaluator_embeddings,
                run_config=run_config,
                raise_exceptions=False,
                show_progress=True,
            )
            df = resultado.to_pandas()
            df.to_csv(ruta_checkpoint, index=False, encoding="utf-8")
            print(f"  [{etiqueta}] listo, checkpoint guardado en {ruta_checkpoint}")
            return df
        except Exception as e:
            print(f"  [{etiqueta}] intento {intento}/{intentos_externos} fallo: {e}")
            if intento < intentos_externos:
                time.sleep(espera_entre_intentos)
    print(f"  [{etiqueta}] se agotaron los reintentos, se omite esta combinacion.")
    return None

In [14]:
resultados_por_combinacion = {}

for c in combinaciones_generadas:
    etiqueta = etiqueta_de(c)
    print(f"\n=== Evaluando: {etiqueta} ({c['total_preguntas']} preguntas) ===")
    df = evaluar_combinacion(etiqueta, c["resultados"])
    if df is not None:
        df["combinacion"] = etiqueta
        df["pipeline_retrieve"] = c["pipeline_retrieve"]
        df["modelo_llm"] = c["modelo_llm"]
        df["estilo_prompt"] = c["estilo_prompt"]
        resultados_por_combinacion[etiqueta] = df

print(f"\nCombinaciones evaluadas con exito: {len(resultados_por_combinacion)} / {len(combinaciones_generadas)}")


=== Evaluando: openai_cs800__raft-llama3.1-8b__raft (196 preguntas) ===


Evaluating:  10%|█         | 81/784 [08:39<6:12:37, 31.80s/it]Exception raised in Job[78]: TimeoutError()
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  11%|█         | 83/784 [30:32<54:04:53, 277.74s/it]Exception raised in Job[83]: TimeoutError()
Exception raised in Job[84]: TimeoutError()
Exception raised in Job[82]: TimeoutError()
Evaluating: 100%|██████████| 784/784 [1:06:28<00:00,  5.09s/it]


  [openai_cs800__raft-llama3.1-8b__raft] listo, checkpoint guardado en validacion\checkpoints_evaluacion_generacion\openai_cs800__raft-llama3.1-8b__raft.csv

=== Evaluando: openai_cs800__raft-llama3.1-8b__estructurado (196 preguntas) ===


Evaluating: 100%|██████████| 784/784 [39:48<00:00,  3.05s/it]


  [openai_cs800__raft-llama3.1-8b__estructurado] listo, checkpoint guardado en validacion\checkpoints_evaluacion_generacion\openai_cs800__raft-llama3.1-8b__estructurado.csv

=== Evaluando: openai_cs800__raft-llama3.1-8b__directo (196 preguntas) ===


Evaluating: 100%|██████████| 784/784 [41:20<00:00,  3.16s/it]


  [openai_cs800__raft-llama3.1-8b__directo] listo, checkpoint guardado en validacion\checkpoints_evaluacion_generacion\openai_cs800__raft-llama3.1-8b__directo.csv

Combinaciones evaluadas con exito: 3 / 3


## B3. Detalle y resumen RAFT

In [21]:
METRICAS_NOMBRES = ["faithfulness", "answer_relevancy", "answer_correctness", "semantic_similarity"]
claves = ["combinacion", "pipeline_retrieve", "modelo_llm", "estilo_prompt"]

df_raft = pd.concat(resultados_por_combinacion.values(), ignore_index=True)

# Marcas de validacion (por combinacion + pregunta)
marcas = val[["combinacion", "pregunta", "id", "falla_modelo", "mod_degenerado", "mod_cortada_por_longitud",
              "mod_formato_no_respetado", "info_abstencion", "info_citas", "info_citas_literales"]]
df_raft = df_raft.merge(marcas, left_on=["combinacion", "user_input"],
                        right_on=["combinacion", "pregunta"], how="left").drop(columns=["pregunta"])
assert df_raft["id"].notna().all(), "Hay filas de RAGAS que no se pudieron emparejar con la validacion"
df_raft.to_excel(r"C:\Users\gomex\OneDrive\Escritorio\TG - Pruebas 2\evaluacion_generacion_detalle_raft.xlsx", index=False)
print(f"Detalle RAFT guardado en {RUTA_SALIDA_DETALLE} ({len(df_raft)} filas)")


def resumir(df):
    columnas = [m for m in METRICAS_NOMBRES if m in df.columns]
    r = df.groupby(claves)[columnas].mean()
    r["n_preguntas"] = df.groupby(claves).size()
    r["metricas_nulas"] = df.groupby(claves)[columnas].apply(lambda x: int(x.isna().sum().sum()))
    r["promedio_4_metricas"] = r[columnas].mean(axis=1)
    return r.reset_index()


resumen_raft = resumir(df_raft).merge(
    resumen_val[["fallas_modelo", "abstenciones", "pct_citas_literales"]],
    left_on="combinacion", right_index=True, how="left",
)

resumen_raft.to_csv(RUTA_SALIDA_RESUMEN, index=False, encoding="utf-8")
print(f"Resumen RAFT guardado en {RUTA_SALIDA_RESUMEN}")
resumen_raft.round(3)

Detalle RAFT guardado en ./validacion/evaluacion_generacion_detalle_raft.csv (588 filas)
Resumen RAFT guardado en ./validacion/evaluacion_generacion_resumen_raft.csv


,combinacion,pipeline_retrieve,modelo_llm,estilo_prompt,faithfulness,answer_relevancy,answer_correctness,semantic_similarity,n_preguntas,metricas_nulas,promedio_4_metricas,fallas_modelo,abstenciones,pct_citas_literales
0,openai_cs800__raft-llama3.1-8b__directo,openai_cs800,raft-llama3.1-8b,directo,0.919,0.774,0.677,0.808,196,2,0.794,1,3,NaN
1,openai_cs800__raft-llama3.1-8b__estructurado,openai_cs800,raft-llama3.1-8b,estructurado,0.911,0.790,0.674,0.798,196,0,0.793,0,3,NaN
2,openai_cs800__raft-llama3.1-8b__raft,openai_cs800,raft-llama3.1-8b,raft,0.873,0.808,0.670,0.812,196,4,0.791,2,0,73.0


## B4. Comparacion con Claude sobre `openai_cs800`

Lee el detalle **ya existente** de Claude (`evaluacion_generacion_detalle.csv`, sin modificarlo ni reevaluarlo) y lo pone junto a RAFT, sobre las mismas 196 preguntas y los mismos contextos. Se calcula dos veces:
- `todas`: el resultado principal.
- `sin_fallas`: excluye las preguntas con fallas del modelo RAFT en **todas** las combinaciones, para mantener la comparacion pareada.

`metricas_nulas` > 0 significa que el juez no pudo puntuar algunas filas (timeouts o errores de parseo). Si es alto en una combinacion RAFT, borra su checkpoint y vuelve a evaluarla.

In [17]:
df_claude = pd.read_csv(RUTA_DETALLE_CLAUDE)
df_claude = df_claude[df_claude["pipeline_retrieve"] == "openai_cs800"]

columnas_comunes = ["user_input"] + claves + [m for m in METRICAS_NOMBRES if m in df_raft.columns]
df_comp = pd.concat([df_claude[columnas_comunes], df_raft[columnas_comunes]], ignore_index=True)
df_comp = df_comp[df_comp["user_input"].isin(preguntas_originales)]

comparacion = pd.concat([
    resumir(df_comp).assign(conjunto="todas"),
    resumir(df_comp[~df_comp["user_input"].isin(preguntas_excluir_sensibilidad)]).assign(conjunto="sin_fallas"),
], ignore_index=True)
comparacion = comparacion.sort_values(["conjunto", "promedio_4_metricas"], ascending=[False, False])
comparacion.to_csv(RUTA_SALIDA_COMPARACION, index=False, encoding="utf-8")
print(f"Comparacion guardada en {RUTA_SALIDA_COMPARACION}")
comparacion[comparacion["conjunto"] == "todas"].round(3)

Comparacion guardada en ./validacion/comparacion_openai_cs800_con_raft.csv


,combinacion,pipeline_retrieve,modelo_llm,estilo_prompt,faithfulness,answer_relevancy,answer_correctness,semantic_similarity,n_preguntas,metricas_nulas,promedio_4_metricas,conjunto
1,openai_cs800__claude-haiku-4.5__estructurado,openai_cs800,claude-haiku-4.5,estructurado,0.959,0.773,0.669,0.799,196,0,0.800,todas
4,openai_cs800__raft-llama3.1-8b__directo,openai_cs800,raft-llama3.1-8b,directo,0.919,0.774,0.677,0.808,196,2,0.794,todas
5,openai_cs800__raft-llama3.1-8b__estructurado,openai_cs800,raft-llama3.1-8b,estructurado,0.911,0.790,0.674,0.798,196,0,0.793,todas
6,openai_cs800__raft-llama3.1-8b__raft,openai_cs800,raft-llama3.1-8b,raft,0.873,0.808,0.670,0.812,196,4,0.791,todas
2,openai_cs800__claude-sonnet-5__directo,openai_cs800,claude-sonnet-5,directo,0.955,0.721,0.681,0.801,196,0,0.789,todas
3,openai_cs800__claude-sonnet-5__estructurado,openai_cs800,claude-sonnet-5,estructurado,0.941,0.701,0.703,0.803,196,0,0.787,todas
0,openai_cs800__claude-haiku-4.5__directo,openai_cs800,claude-haiku-4.5,directo,0.948,0.765,0.647,0.781,196,1,0.785,todas


In [18]:
comparacion[comparacion["conjunto"] == "sin_fallas"].round(3)

,combinacion,pipeline_retrieve,modelo_llm,estilo_prompt,faithfulness,answer_relevancy,answer_correctness,semantic_similarity,n_preguntas,metricas_nulas,promedio_4_metricas,conjunto
8,openai_cs800__claude-haiku-4.5__estructurado,openai_cs800,claude-haiku-4.5,estructurado,0.959,0.777,0.670,0.800,194,0,0.802,sin_fallas
11,openai_cs800__raft-llama3.1-8b__directo,openai_cs800,raft-llama3.1-8b,directo,0.919,0.778,0.679,0.809,194,0,0.796,sin_fallas
12,openai_cs800__raft-llama3.1-8b__estructurado,openai_cs800,raft-llama3.1-8b,estructurado,0.913,0.790,0.677,0.800,194,0,0.795,sin_fallas
13,openai_cs800__raft-llama3.1-8b__raft,openai_cs800,raft-llama3.1-8b,raft,0.880,0.810,0.674,0.815,194,4,0.795,sin_fallas
9,openai_cs800__claude-sonnet-5__directo,openai_cs800,claude-sonnet-5,directo,0.956,0.726,0.682,0.802,194,0,0.791,sin_fallas
10,openai_cs800__claude-sonnet-5__estructurado,openai_cs800,claude-sonnet-5,estructurado,0.946,0.703,0.709,0.806,194,0,0.791,sin_fallas
7,openai_cs800__claude-haiku-4.5__directo,openai_cs800,claude-haiku-4.5,directo,0.947,0.769,0.648,0.782,194,1,0.787,sin_fallas


## Notas

- **Comparabilidad.** El juez, los embeddings, las metricas, las 196 preguntas y los contextos (`openai_cs800`) son los mismos de la evaluacion de Claude. La unica diferencia en la generacion es el modelo, que corre en local y no en OpenRouter.
- **Que se evalua en RAFT.** `respuesta_generada` contiene solo lo que va despues de `##Answer:`, igual que en `estructurado` se evalua solo lo que va despues de `Respuesta final:`. El razonamiento con citas no entra en las metricas, pero queda en `respuesta_generada_cruda`.
- **Costo.** Solo se evaluan las 3 combinaciones RAFT (~588 filas). Haz primero una prueba con `TAMANO_MUESTRA = 15` para calibrar.
- **Siguiente paso.** `evaluacion_generacion_detalle_raft.csv` tiene las mismas columnas que `evaluacion_generacion_detalle.csv`, mas las marcas al final. Se pueden concatenar para las pruebas estadisticas pareadas, por ejemplo RAFT-raft frente a Sonnet-estructurado sobre las mismas 196 preguntas.